# Module 5.1 — Threading Fundamentals

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

Concurrency means structuring a program so that several tasks make progress in overlapping time periods. Threads are one way to achieve this, and they are especially useful for input/output work such as network requests and file access. This module covers creating threads, coordinating them safely, and the thread-safe queue pattern.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it (`Shift + Enter`).
- Because threads run concurrently, raw print order can vary; the examples therefore collect results and report them after the threads finish, so the output is stable.
- Attempt the exercises before consulting the solutions at the end.

### Learning objectives

After completing this module you will be able to:

1. Create and run threads with `threading.Thread`, `start()`, and `join()`.
2. Explain why shared mutable state needs protection, and use a `Lock`.
3. Use other synchronisation primitives: `RLock`, `Semaphore`, `Event`, and `Condition`.
4. Apply the producer-consumer pattern with a thread-safe `queue.Queue`.
5. Understand the Global Interpreter Lock and when threads help.

**Prerequisites:** Tracks 1 to 4.

---

## Part 1 · Creating and Running Threads

A thread is an independent line of execution within a program. The `threading` module lets you run a function in a separate thread: construct a `Thread` with the target function and its arguments, call `start()` to begin it, and call `join()` to wait for it to finish.

Threads are well suited to **input/output-bound** work, where the program spends time waiting (for a network reply, a disk read, a timer). While one thread waits, others can proceed.

In [ ]:
import threading
import time

results = []                       # threads will record their outcomes here

def worker(name, delay):
    time.sleep(delay)              # simulate waiting for input/output
    results.append(f"{name} finished after {delay}s")

# Create several threads, each running 'worker' with different arguments.
threads = [
    threading.Thread(target=worker, args=("A", 0.03)),
    threading.Thread(target=worker, args=("B", 0.01)),
    threading.Thread(target=worker, args=("C", 0.02)),
]

for t in threads:
    t.start()                      # begin each thread
for t in threads:
    t.join()                       # wait for all to complete

# Sorted so the printed output is stable regardless of finishing order.
print("\n".join(sorted(results)))

Running tasks concurrently can be much faster than one after another when the work is mostly waiting. Three tasks that each wait a moment finish in about the time of the longest one, not the sum of them all, because the waiting overlaps.

In [ ]:
import threading
import time

def wait_task(duration):
    time.sleep(duration)

durations = [0.05, 0.05, 0.05]

# Sequential: each task runs after the previous one.
start = time.perf_counter()
for d in durations:
    wait_task(d)
sequential = time.perf_counter() - start

# Concurrent: all tasks wait at the same time.
start = time.perf_counter()
threads = [threading.Thread(target=wait_task, args=(d,)) for d in durations]
for t in threads: t.start()
for t in threads: t.join()
concurrent = time.perf_counter() - start

print(f"sequential took about {sequential:.2f}s")
print(f"concurrent took about {concurrent:.2f}s")
print("concurrent is roughly the time of one task, not three")

## Part 2 · Shared State and the `Lock`

When several threads modify the same data, their operations can interleave in unsafe ways, producing a **race condition** and a wrong result. Even a simple `counter += 1` is really read-modify-write, three steps that another thread can interrupt. A `Lock` ensures that only one thread at a time runs the protected section, making updates safe.

In [ ]:
import threading
import time

# An unsafe counter: read-modify-write with no protection.
unsafe_counter = 0
def increment_unsafe():
    global unsafe_counter
    for _ in range(50_000):
        current = unsafe_counter
        unsafe_counter = current + 1     # another thread may interleave here

threads = [threading.Thread(target=increment_unsafe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("unsafe result:", unsafe_counter, "(expected 200000; usually less due to races)")

In [ ]:
import threading

# A safe counter: the lock guarantees one update at a time.
safe_counter = 0
lock = threading.Lock()

def increment_safe():
    global safe_counter
    for _ in range(50_000):
        with lock:                       # only one thread holds the lock at a time
            safe_counter += 1

threads = [threading.Thread(target=increment_safe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("safe result:", safe_counter, "(always exactly 200000)")

## Part 3 · Other Synchronisation Primitives

Beyond the basic `Lock`, the `threading` module offers more specialised tools:

- `RLock` (reentrant lock) can be acquired again by the same thread that already holds it, which a plain `Lock` cannot.
- `Semaphore` allows up to a fixed number of threads into a section at once, useful for limiting concurrency.
- `Event` lets one thread signal others to proceed.
- `Condition` lets threads wait until notified that some state has changed.

In [ ]:
import threading
import time

# Semaphore: allow at most 2 threads into the "service" at once.
semaphore = threading.Semaphore(2)
log = []
log_lock = threading.Lock()

def use_service(name):
    with semaphore:                      # blocks if 2 threads are already inside
        with log_lock:
            log.append(f"{name} entered")
        time.sleep(0.02)
        with log_lock:
            log.append(f"{name} left")

threads = [threading.Thread(target=use_service, args=(f"T{i}",)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()

# Count is deterministic even though exact order varies.
entered = sum(1 for line in log if "entered" in line)
print("total entries recorded:", entered)
print("at most 2 were ever inside at once (enforced by the semaphore)")

In [ ]:
import threading
import time

# Event: a 'start signal' that worker threads wait for.
start_signal = threading.Event()
order = []
order_lock = threading.Lock()

def runner(name):
    start_signal.wait()                  # block until the event is set
    with order_lock:
        order.append(name)

threads = [threading.Thread(target=runner, args=(f"R{i}",)) for i in range(3)]
for t in threads: t.start()

time.sleep(0.02)                          # threads are now all waiting
print("workers are waiting:", not start_signal.is_set())
start_signal.set()                        # release them all at once

for t in threads: t.join()
print("all", len(order), "runners proceeded after the signal")

## Part 4 · The Producer-Consumer Pattern with `queue.Queue`

A very common design has some threads **producing** work items and others **consuming** them. A `queue.Queue` is built for exactly this: it is thread-safe (no manual locking needed), and consumers can block waiting for items. A sentinel value (a marker) is a clean way to tell consumers to stop.

In [ ]:
import threading
import queue

work = queue.Queue()
results = []
results_lock = threading.Lock()
SENTINEL = None                          # a marker meaning "no more work"

def producer():
    for i in range(1, 6):
        work.put(i)                      # add items to the queue
    work.put(SENTINEL)                   # signal the consumer to stop

def consumer():
    while True:
        item = work.get()                # blocks until an item is available
        if item is SENTINEL:
            break
        with results_lock:
            results.append(item * item)  # process the item
        work.task_done()

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start(); c.start()
p.join(); c.join()

print("processed results:", sorted(results))

## Part 5 · The Global Interpreter Lock

A crucial fact about standard CPython: the **Global Interpreter Lock** (GIL) allows only one thread to execute Python bytecode at a time. This means threads do **not** speed up CPU-bound work (heavy calculation), because they cannot truly run Python code in parallel. Threads do help with **input/output-bound** work, because the GIL is released while a thread waits on input/output, letting others run.

The practical guidance: use threads for input/output-bound concurrency; for CPU-bound parallelism, use multiprocessing (Module 5.2). Recent Python versions are introducing an experimental free-threaded build that removes the GIL, discussed in Track 7.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Running a function in a thread (Easy)

In [ ]:
import threading

output = []
def task(label):
    output.append(f"task {label} done")

t = threading.Thread(target=task, args=("X",))
t.start()
t.join()                       # wait for it to finish before reading the result
print(output)

### Example 2 — Several threads, collected results (Easy)

In [ ]:
import threading

squares = []
lock = threading.Lock()

def square(n):
    with lock:
        squares.append(n * n)

threads = [threading.Thread(target=square, args=(n,)) for n in range(1, 6)]
for t in threads: t.start()
for t in threads: t.join()
print(sorted(squares))

### Example 3 — Protecting a shared total with a lock (Medium)

In [ ]:
import threading

balance = 0
lock = threading.Lock()

def deposit(amount, times):
    global balance
    for _ in range(times):
        with lock:
            balance += amount

threads = [
    threading.Thread(target=deposit, args=(10, 1000)),
    threading.Thread(target=deposit, args=(5, 1000)),
]
for t in threads: t.start()
for t in threads: t.join()
print("final balance:", balance)          # 10*1000 + 5*1000 = 15000, always

### Example 4 — Limiting concurrency with a semaphore (Medium)

In [ ]:
import threading
import time

# Allow only 3 downloads at a time, simulating a connection limit.
limit = threading.Semaphore(3)
completed = []
lock = threading.Lock()

def download(item):
    with limit:
        time.sleep(0.01)
        with lock:
            completed.append(item)

threads = [threading.Thread(target=download, args=(f"file{i}",)) for i in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print("downloaded count:", len(completed))

### Example 5 — Producer-consumer with multiple consumers (Difficult)

In [ ]:
import threading
import queue

work = queue.Queue()
results = []
results_lock = threading.Lock()

def producer(n):
    for i in range(1, n + 1):
        work.put(i)

def consumer():
    while True:
        try:
            item = work.get(timeout=0.1)     # stop if no work arrives soon
        except queue.Empty:
            return
        with results_lock:
            results.append(item * 2)
        work.task_done()

prod = threading.Thread(target=producer, args=(10,))
prod.start(); prod.join()                    # fill the queue first

consumers = [threading.Thread(target=consumer) for _ in range(3)]
for c in consumers: c.start()
for c in consumers: c.join()

print("count:", len(results), "| total:", sum(results))

### Example 6 — Coordinating phases with an Event (Difficult)

In [ ]:
import threading
import time

ready = threading.Event()
phase_log = []
log_lock = threading.Lock()

def participant(name):
    with log_lock:
        phase_log.append(f"{name} preparing")
    ready.wait()                              # all wait here for the go signal
    with log_lock:
        phase_log.append(f"{name} running")

threads = [threading.Thread(target=participant, args=(f"P{i}",)) for i in range(3)]
for t in threads: t.start()

time.sleep(0.02)
preparing = sum(1 for line in phase_log if "preparing" in line)
print("all prepared before signal:", preparing == 3)

ready.set()                                   # release everyone together
for t in threads: t.join()
running = sum(1 for line in phase_log if "running" in line)
print("all ran after signal:", running == 3)

---

## Exercises

**Exercise 1 (Easy).** Create three threads, each running a function that appends its name to a shared list (protected by a lock). Start and join them, then print the sorted list.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a function `count_to(n)` and run it in a single thread, joining before printing a confirmation message.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Two threads each add 1 to a shared counter 100,000 times. Use a `Lock` so the final value is exactly 200,000, and print it.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use a `queue.Queue` with one producer that puts the numbers 1 to 5 plus a sentinel `None`, and one consumer that collects the cubes of the numbers. Print the collected list.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Use a `Semaphore` that permits 2 concurrent workers. Launch 6 workers, each appending a record to a shared list. Print how many records were collected (it should be 6).

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import threading
names = []
lock = threading.Lock()

def record(name):
    with lock:
        names.append(name)

threads = [threading.Thread(target=record, args=(n,)) for n in ["A", "B", "C"]]
for t in threads: t.start()
for t in threads: t.join()
print(sorted(names))

**Solution 2**

In [ ]:
import threading

def count_to(n):
    total = sum(range(1, n + 1))
    print("counted to", n, "sum is", total)

t = threading.Thread(target=count_to, args=(100,))
t.start()
t.join()
print("thread finished")

**Solution 3**

In [ ]:
import threading
counter = 0
lock = threading.Lock()

def add():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1

threads = [threading.Thread(target=add) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print(counter)

**Solution 4**

In [ ]:
import threading, queue
q = queue.Queue()
cubes = []

def producer():
    for i in range(1, 6):
        q.put(i)
    q.put(None)

def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        cubes.append(item ** 3)

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start(); c.start(); p.join(); c.join()
print(cubes)

**Solution 5**

In [ ]:
import threading, time
sem = threading.Semaphore(2)
records = []
lock = threading.Lock()

def worker(i):
    with sem:
        time.sleep(0.01)
        with lock:
            records.append(i)

threads = [threading.Thread(target=worker, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()
print("records collected:", len(records))

---

## Summary and Key Points

- A thread runs a function concurrently; create it with `threading.Thread(target=...)`, begin with `start()`, and wait with `join()`. Threads excel at input/output-bound work because waiting overlaps.
- Shared mutable state can suffer race conditions because operations interleave; a `Lock` (used with `with`) makes a section run one thread at a time, producing correct results.
- Other primitives serve specific needs: `RLock` (reacquirable by its holder), `Semaphore` (limit concurrency), `Event` (signal others to proceed), and `Condition` (wait for a state change).
- `queue.Queue` is a thread-safe channel for the producer-consumer pattern; a sentinel value cleanly signals consumers to stop.
- The Global Interpreter Lock allows only one thread to run Python bytecode at a time, so threads do not speed up CPU-bound work; use multiprocessing for that, covered next.

### Next module: 5.2 — Multiprocessing

The next module covers running work in separate processes to achieve true parallelism for CPU-bound tasks, with process pools, shared data, and the pitfalls to avoid.